# 🌸 Scikit-learn ONNX Inference with SnoozeML

This notebook demonstrates the end-to-end workflow:
1. **Train** a `RandomForestClassifier` on the classic Iris dataset.
2. **Export** the model to ONNX using `snoozeml.converters.export_sklearn`.
3. **Upload & Deploy** the model directly to SnoozeML via **HTTP multipart upload** (no volume mounting or daemon restart needed!).
4. **Cold Start vs. Warm Execution**: Benchmark cold-start loading vs. ultra-low latency in-memory inference.

In [1]:
from pathlib import Path
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from snoozeml import SnoozeClient
from snoozeml.converters import export_sklearn

## 1. Load Data & Train Model

In [2]:
# Load the classic Iris dataset (4 features, 3 classes) and split 80/20
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit a Random Forest classifier
clf = RandomForestClassifier(n_estimators=20, random_state=42).fit(X_train, y_train)
print(f"Training accuracy: {clf.score(X_train, y_train):.4f}")

Training accuracy: 1.0000


## 2. Export & Upload Directly to SnoozeML via HTTP

Rather than manually mounting container volumes or restarting the daemon, you can upload models directly via HTTP using `client.upload_model()`. You can also configure runtime parameters such as dynamic batch size, batch delay budget, and scale-to-zero idle timeout.

In [3]:
# Export scikit-learn model locally to single-tensor FP32 ONNX
local_onnx = Path("./iris_classifier.onnx")
export_sklearn(clf, sample_input=X_train, input_name="input", output_path=local_onnx)
print(f"Exported model locally to {local_onnx} ({local_onnx.stat().st_size} bytes)")

# Connect to SnoozeML daemon
# If authentication is enabled on the server, supply your API key or JWT token:
# client = SnoozeClient(endpoint="http://localhost:8080", api_key="snooze_admin_secret_key")
client = SnoozeClient(endpoint="http://localhost:8080")

# Upload directly via HTTP with custom runtime configuration (no volume mount or restart required!)
model_info = client.upload_model(
    file=local_onnx,
    model_id="iris_classifier",
    description="Scikit-learn Random Forest on Iris Dataset",
    pinned=False,               # Keep unpinned to allow scale-to-zero memory reclamation
    max_batch_size=16,          # Dynamic micro-batching up to 16 requests
    max_batch_delay_ms=10,      # 10ms batch latency budget
    idle_timeout_secs=300,      # Deallocate after 5 minutes of inactivity
)
print(f"Uploaded model: {model_info.id} | Initial State: {model_info.state.value} | Pinned: {model_info.pinned}")

# Note: You can also trigger a runtime refresh at any time to re-scan the models directory:
# client.refresh_models()

Exported model locally to iris_classifier.onnx (53342 bytes)
Uploaded model: iris_classifier | Initial State: idle | Pinned: False


## 3. Compare Cold Start vs. Warm In-Memory Inference

SnoozeML automatically loads weights on the first incoming request (**cold start**), keeps them warm in memory for subsequent traffic, and unloads them when idle (**scale-to-zero**).

In [4]:
# Ensure model starts in idle state (scale-to-zero)
client.models.evict("iris_classifier")
print(f"State before 1st request: {client.models.get('iris_classifier').state.value}")

# --- 1st Request: Cold Start (eager weight loading into memory + forward pass) ---
resp_cold = client.infer(model_id="iris_classifier", inputs={"input": X_test})
print(f"❄️  Cold Start Latency: {resp_cold.duration_ms:.2f} ms (weights loaded on-demand + forward pass)")
print(f"State after 1st request:  {client.models.get('iris_classifier').state.value}")

# --- 2nd Request: Warm Execution (model already resident in memory) ---
resp_warm = client.infer(model_id="iris_classifier", inputs={"input": X_test})
print(f"🔥 Warm Latency:        {resp_warm.duration_ms:.2f} ms (ultra-fast in-memory inference)")

speedup = resp_cold.duration_ms / max(resp_warm.duration_ms, 0.01)
print(f"⚡ Speedup:             {speedup:.1f}x faster warm execution")

State before 1st request: idle
❄️  Cold Start Latency: 146.62 ms (weights loaded on-demand + forward pass)
State after 1st request:  ready
🔥 Warm Latency:        1.31 ms (ultra-fast in-memory inference)
⚡ Speedup:             112.0x faster warm execution


In [5]:
# Evaluate predictions against ground truth
probs = resp_warm.get_numpy() # shape: (30, 3)
preds = probs.argmax(axis=1)

print(f"SnoozeML Test Accuracy:  {accuracy_score(y_test, preds):.4f}")
print(f"Predicted classes:       {preds[:5]}")
print(f"Ground truth classes:    {y_test[:5]}")

SnoozeML Test Accuracy:  1.0000
Predicted classes:       [1 0 2 1 1]
Ground truth classes:    [1 0 2 1 1]
